<a href="https://colab.research.google.com/github/MaryObr/humor_diploma/blob/main/final_data_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Dependencies

In [ ]:
!pip install telethon nest_asyncio
!pip install concurrent.futures tqdm -q

In [ ]:
from tqdm import tqdm
from telethon import TelegramClient
from telethon.tl.types import DocumentAttributeAudio
from google.colab import userdata
from tqdm import tqdm
import nest_asyncio
import os
import json
from datetime import datetime
from google.colab import files
import requests
import json
import os
from google.colab import userdata
from tqdm import tqdm
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor
import time
from disk_interactions import *
nest_asyncio.apply()

### Подкасты

Источник -- https://dostoverno.ru

In [ ]:
def dosto_data(query:str):
  ''''
  Функция для скрейпинга подкастов с сайта https://dostoverno.ru.
  Принимает ссылку на серию подкастов, выгружает аудиофайлы на Яндекс Диск
  Аргументы:
    query -- абсолютная ссылка на страницу с подкастом
  '''
  podcast_name = query.split('/')[-1]
  data = requests.get(query)
  data = soup(data.text,"html5lib")

  div = data.find('div', {'id': 'amplitude-player'})
  link = div.get('data-playlist')

  links = link.split('http')

  # для каждой аудиозаписи
  for l in tqdm(range(len(links))):

    if 'mp3' in links[l]:

      links[l] = 'http'+links[l].split('mp3')[0]+'mp3'
      links[l] = links[l].replace('\\/', '/')

      # скачиваем файл
      mp3_response = requests.get(links[l], stream=True)
      with open(f'{podcast_name}-{l}.mp3', 'wb') as f:
        for chunk in mp3_response.iter_content(chunk_size=8192):
            f.write(chunk)

      # загружаем его на диск
      upload_file(f'{podcast_name}-{l}.mp3', 'negative_data')

In [ ]:
# лекции служители науки

dosto_data('https://dostoverno.ru/podcasts/sluzitel-nauki')

In [ ]:
# лекции стиль жизни

dosto_data('https://dostoverno.ru/podcasts/podkasty-o-stile-zizni')

In [ ]:
# лекции о моде

dosto_data('https://dostoverno.ru/podcasts/podkasty-o-mode')

In [ ]:
# лекции об искусстве

dosto_data('https://dostoverno.ru/podcasts/podkasty-ob-iskusstve')

### Стэндапы

Источник -- https://t.me/audiostandup

In [ ]:
api_id = userdata.get('telegram_id')
api_hash = userdata.get('telegram_hash')

In [ ]:
async def main(channel_name:str, save_dir:str, last_saved:int=0):
    '''
    Асинхронная функция для скачивания записей из телеграм канала https://t.me/audiostandup, сохраняет .mp3 файлы и файл логирования
    Аргументы:
      channel_name -- название канала
      save_dir -- локальный путь к папке, в которую сохранятся данные
      last_saved -- id последнего скачанного файла на случай возобновления скачивания
    '''
    client = TelegramClient('standup', api_id, api_hash)
    await client.start()

    download_dir = '/content/'

    channel = await client.get_entity(channel_name)

    offset_id = 0
    if last_saved!=offset_id:
      offset_id = last_saved

    # общее число сообщений для tqdm
    total_messages = await client.get_messages(channel, limit=0)
    total_count = total_messages.total

    file_path = ''
    # Скачивание только аудиофайлов
    with tqdm(total=total_count, desc="Download & upload", unit="msg") as pbar:
      async for message in client.iter_messages(channel, offset_id=offset_id):

          if message.audio:
              file_path = await message.download_media(download_dir)

          elif message.document:
              # Проверка на mp3 формат
              doc = message.document
              if doc.mime_type == 'audio/mpeg' or doc.attributes:
                  for attr in doc.attributes:
                      if isinstance(attr, DocumentAttributeAudio):
                          file_path = await message.download_media(download_dir) # скачиваем
                          break

          if file_path !='':
            upload_file(str(file_path).strip('/content/'), save_dir)

          # логируем последний обработанный файл на случай падения сессии
          log_info ={
              'last_processed_id': message.id,
              'timestamp': datetime.now().isoformat()
          }

          with open('log_file.json', 'w') as log_file:
            json.dump(log_info, log_file, indent=2)

          if (total_count-int(message.id)) % 10 == 0:
            files.download('log_file.json')

          pbar.update(1)

await main('https://t.me/audiostandup', 'standup', 2276)